# Feature Engineering — Kelmarsh Wind Farm (Turbine 1)

This notebook prepares the clean SCADA data for the LSTM autoencoder model.

**Goals:**
- Add turbine identifier for multi-turbine extension
- Normalize all features to [0, 1] range
- Build sliding windows of 30 timesteps (5 hours) for sequence modeling
- Split into train/test sets preserving temporal order

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import joblib
import os

print("Libraries imported")

Libraries imported


## 1. Load Clean Data

Loading the cleaned dataset produced in `01_eda.ipynb`.

In [24]:
df = pd.read_csv(
    '../data/processed/turbine1_clean.csv',
    index_col='timestamp',
    parse_dates=True
)

print(f"Shape: {df.shape}")
df.head(3)

Shape: (46694, 9)


,wind_speed,wind_dir,power,rotor_rpm,nacelle_pos,gen_bearing_front_temp,gen_bearing_rear_temp,gear_oil_temp,pitch_angle
timestamp,,,,,,,,,
2020-01-01 00:00:00,3.887291,116.839462,150.984141,8.449412,97.146416,38.2775,36.752500,56.972500,0.049498
2020-01-01 00:10:00,3.848941,115.659361,130.414782,8.345387,97.146416,39.1450,37.037499,56.695001,0.074499
2020-01-01 00:20:00,4.043625,116.646515,146.401656,8.497044,100.748278,39.8975,37.347500,56.640000,0.074499


## 2. Add Turbine ID

Adding a turbine identifier column. This will allow us to combine 
all 6 turbines into a single dataset in a later stage.

In [25]:
df['turbine_id'] = 1

print(df.shape)
df['turbine_id'].value_counts()

(46694, 10)


turbine_id
1    46694
Name: count, dtype: int64

## 3. Normalize Features

LSTM networks are sensitive to input scale. We apply MinMaxScaler 
to bring all features into the [0, 1] range.

The scaler is saved to `data/processed/scaler.pkl` so it can be 
reused at inference time to inverse transform predictions.

In [26]:
# Columns to scale (everything except turbine_id)
feature_cols = ['wind_speed', 'wind_dir', 'power', 'rotor_rpm', 
                'nacelle_pos', 'gen_bearing_front_temp', 
                'gen_bearing_rear_temp', 'gear_oil_temp', 'pitch_angle']

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

# Save the scaler — we'll need it later to inverse transform predictions
os.makedirs('../data/processed', exist_ok=True)
joblib.dump(scaler, '../data/processed/scaler.pkl')

df_scaled[feature_cols].describe().round(3)

,wind_speed,wind_dir,power,rotor_rpm,nacelle_pos,gen_bearing_front_temp,gen_bearing_rear_temp,gear_oil_temp,pitch_angle
count,46694.000,46694.000,46694.000,46694.000,46694.000,46694.000,46694.000,46694.000,46694.000
mean,0.273,0.552,0.385,0.757,0.551,0.538,0.507,0.863,0.018
std,0.136,0.248,0.323,0.195,0.248,0.097,0.073,0.075,0.050
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
25%,0.172,0.426,0.106,0.569,0.422,0.475,0.460,0.812,0.000
50%,0.253,0.607,0.281,0.773,0.607,0.557,0.496,0.870,0.000
75%,0.354,0.713,0.634,0.966,0.712,0.611,0.537,0.924,0.010
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


## 4. Build Sliding Windows

The LSTM autoencoder expects sequences, not individual rows. We create 
overlapping windows of 30 consecutive timesteps (5 hours of operation).

Each window captures enough temporal context to learn normal behavioral 
patterns across varying wind conditions.

In [27]:
def create_sequences(data, window_size=30):
    """
    Convert time series into overlapping windows.
    Each sample = 30 consecutive timesteps (= 5 hours of data).
    """
    X = []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
    return np.array(X)

# Use only the feature columns (no turbine_id)
data_array = df_scaled[feature_cols].values

X = create_sequences(data_array, window_size=30)

print(f"Shape of X: {X.shape}")
print(f"Each sample: {X.shape[1]} timesteps x {X.shape[2]} features")


Shape of X: (46664, 30, 9)
Each sample: 30 timesteps x 9 features


## 5. Train / Test Split

We split chronologically — first 80% for training, last 20% for testing.
Random splitting is not used because it would cause data leakage in time series.

In [28]:
# 80% train, 20% test — split by time, not randomly
split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

# Save for use in training notebook
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy', X_test)

X_train: (37331, 30, 9)
X_test:  (9333, 30, 9)


## 6. Save Processed Data

Saving the numpy arrays to `data/processed/` for use in the model training notebook.
Note: `.npy` files are excluded from version control due to size.

In [29]:
import os

files = os.listdir('../data/processed/')
print("Files in data/processed/:")
for f in files:
    size = os.path.getsize(f'../data/processed/{f}') / 1024 / 1024
    print(f"  {f} — {size:.1f} MB")

Files in data/processed/:
  turbine1_clean.csv — 7.8 MB
  scaler.pkl — 0.0 MB
  X_train.npy — 76.9 MB
  X_test.npy — 19.2 MB
